> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


## 实验三：SwiGLU基础版算子开发与验证


建议学时：4学时


## 实验任务


1.任务描述


本实验实现一个采用FP32数据类型的SwiGLU基础版算子，并将其注册为PyTorch可调用接口。基础版以公式正确和工程链路打通为主要目标，代码直接采用全局内存标量读写，便于理解AscendC Kernel、tiling传参、动态库加载和模型替换流程。


2.学习目标


完成本实验后，学生应能够：


• 理解SwiGLU在Transformer前馈网络中的作用和计算公式。


• 掌握逐元素算子的任务划分、全局内存访问和输出写回方式。


• 能够构建AscendC Kernel并通过torch.library注册为PyTorch接口。


• 能够使用单算子测试、独立直调测试和Qwen2.5模型替换测试验证结果。


## 任务准备


1.算子定义


SwiGLU基础版算子接收两路形状一致的输入张量gate和up，输出与输入形状一致。该算子的计算公式如下。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">说明</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">算子名称</td>
<td style="text-align:left;">SwiGLU基础版算子</td>
</tr>
<tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">gate和up，二者形状一致，均为float32类型NPU连续张量</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">output，形状与输入一致</td>
</tr>
<tr>
<td style="text-align:left;">计算语义</td>
<td style="text-align:left;">对每个位置独立计算SiLU门控激活并与up逐元素相乘</td>
</tr>
<tr>
<td style="text-align:left;">模型位置</td>
<td style="text-align:left;">用于替换Qwen2.5 MLP中的激活融合部分</td>
</tr>
</tbody></table>


2.SwiGLU简介


SwiGLU是一类门控激活计算，常见于Qwen、LLaMA等大模型的前馈网络。输入通常来自两路线性投影：一路作为gate，另一路作为up。gate先经过SiLU激活，再与up逐元素相乘，计算公式如下。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">hidden states ├── gate_proj ── SiLU ──┐ └── up_proj ──────────┼── elementwise multiply ── down_proj</th>
</tr>
</thead>
</table>


基础版不改变模型结构，只把图中的SiLU+逐元素乘法替换为自定义算子。它的价值在于建立可构建、可加载、可验证的最小闭环，为后续向量化和流水优化提供基线。


3.算子与接口定义


本实验通过AscendC实现device侧kernel，再通过torch.library注册为PyTorch可调用接口。下表给出本实验中算子接口、入口函数和主要文件之间的对应关系。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">接口项</th>
<th style="text-align:left;">约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">PyTorch调用接口</td>
<td style="text-align:left;">torch.ops.swiglu_custom.swiglu(gate, up)</td>
</tr>
<tr>
<td style="text-align:left;">device侧入口</td>
<td style="text-align:left;">swiglu_baseline_kernel</td>
</tr>
<tr>
<td style="text-align:left;">核心实现文件</td>
<td style="text-align:left;">op_kernel/swiglu_baseline_kernel.h</td>
</tr>
<tr>
<td style="text-align:left;">任务划分文件</td>
<td style="text-align:left;">op_kernel/swiglu_tiling.h</td>
</tr>
<tr>
<td style="text-align:left;">注册文件</td>
<td style="text-align:left;">torch_extension/swiglu_torch_register.asc</td>
</tr>
</tbody></table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">gate和up为形状一致的NPU连续张量</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">与输入形状一致的float32张量</td>
</tr>
<tr>
<td style="text-align:left;">计算公式</td>
<td style="text-align:left;">见前置知识中的公式说明</td>
</tr>
<tr>
<td style="text-align:left;">调用方式</td>
<td style="text-align:left;">torch.ops.swiglu_custom.swiglu(gate, up)</td>
</tr>
<tr>
<td style="text-align:left;">实现特点</td>
<td style="text-align:left;">全局内存标量读写，逻辑清晰，便于理解</td>
</tr>
</tbody></table>


4.实验环境准备


本实验在Ascend NPU云服务器上完成，使用CANN工具链、AscendC和PyTorch NPU环境。基础版与优化版建议放在不同工程目录中，构建和测试也尽量在新的Python进程中执行，避免torch.library重复注册。


进入工程目录后，先加载CANN环境变量，再检查环境、构建工程并设置动态库搜索路径。CANN安装路径以云服务器实际配置为准。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /home/user/SwiGluBaselineExperiment source /usr/local/Ascend/ascend-toolkit/set_env.sh bash scripts/check_env.sh bash scripts/build.sh export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH</th>
</tr>
</thead>
</table>


## 任务实施


## 步骤一：准备工程结构


本步骤介绍工程在文件层面的组织方式。基础版工程将算子内核、PyTorch注册、独立直调、脚本和测试分开存放，使host侧、device侧和模型侧职责清晰。复制工程后统一修改命名，是为了让构建产物、Python接口和运行脚本始终对应同一套算子实现。


基础版工程放在/home/user/SwiGluBaselineExperiment。工程中保留算子源码、PyTorch注册、独立直调程序、构建脚本和测试脚本。复制工程后要同步修改算子内核名称、动态库名称、launch头文件和Python加载路径，避免测试时仍加载旧工程的动态库。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">SwiGluBaselineExperiment/ ├── op_kernel/ │ ├── swiglu_baseline_kernel.cpp │ ├── swiglu_baseline_kernel.h │ └── swiglu_tiling.h ├── torch_extension/ │ ├── <strong>init</strong>.py │ └── swiglu_torch_register.asc ├── verify_kernel_launch/ ├── scripts/ └── tests/</th>
</tr>
</thead>
</table>


## 步骤二：定义任务划分信息


本步骤说明SwiGLU基础版如何把一维输入划分给多个算核。host侧把总元素数、算核数量和每个算核负责的长度打包成tiling数据，再由kernel入口读取后分发到Process()。这样做的作用是把高层输入规模转换成device侧可以直接执行的工作范围。


SwiGLU是逐元素算子，算子内核只需要知道总元素个数、实际启用的算核数量以及每个算核负责的元素段长度。基础版使用16字节的SwiGluTiling结构体传参，字段顺序要与host侧、device侧保持一致。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct SwiGluTiling { uint32_t totalSize = 0; uint32_t coreNum = 1; uint32_t elementsPerCore = 0; uint32_t reserved = 0; }; #pragma pack(pop) static_assert(sizeof(SwiGluTiling) == 16, &quot;SwiGluTiling size must be 16 bytes&quot;);</th>
</tr>
</thead>
</table>


算子内核入口先把全局内存中的tiling数据拷贝到本地结构体，再创建基础版实现对象。这样写的好处是入口函数职责清楚，主体计算集中在Process()中，入口函数只负责准备运行上下文。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">extern &quot;C&quot; <strong>global</strong> <strong>aicore</strong> void swiglu_baseline_kernel( GM_ADDR gate, GM_ADDR up, GM_ADDR output, GM_ADDR workspace, GM_ADDR tiling) { (void)workspace; SwiGluTiling t; const <strong>gm</strong> uint32_t *src = reinterpret_cast&lt;const __gm__ uint32_t *&gt;(tiling); uint32_t *dst = reinterpret_cast&lt;uint32_t *&gt;(&amp;t); for (uint32_t i = 0; i &lt; sizeof(SwiGluTiling) / sizeof(uint32_t); ++i) { dst[i] = src[i]; } KernelSwiGluBaseline op; op.Init(gate, up, output, t.totalSize, t.coreNum, t.elementsPerCore); op.Process(); }</th>
</tr>
</thead>
</table>


## 步骤三：实现基础版逐元素计算


本步骤展示基础版SwiGLU的核心公式如何落到device代码里。kernel直接从GM读取gate和up，依次完成sigmoid、SiLU和逐元素乘法，再把结果写回GM。它体现的是“公式驱动实现”的思路，逻辑最直观，也最适合建立正确性基线。


基础版直接在全局内存上读取gate和up。每个算核按照begin到end处理一段连续元素，单个元素的计算遵循前置知识中给出的SwiGLU公式。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;"><strong>aicore</strong> inline void Process() { const uint32_t coreId = GetBlockIdx(); if (coreId &gt;= coreNum_) { return; } const uint32_t begin = coreId * elementsPerCore_; uint32_t end = begin + elementsPerCore_; if (end &gt; totalSize_) { end = totalSize_; } for (uint32_t i = begin; i &lt; end; ++i) { const float gate = gateGm_.GetValue(i); const float up = upGm_.GetValue(i); const float sigmoid = 1.0f / (1.0f + SwigluExpApprox(-gate)); outputGm_.SetValue(i, gate * sigmoid * up); } }</th>
</tr>
</thead>
</table>


## 步骤四：注册PyTorch调用接口


本步骤把自定义算子接入PyTorch调用链。代码先完成输入合法性约束，再分配输出张量、申请tiling和workspace、获取当前NPU流，最后通过ACLRT_LAUNCH_KERNEL启动算子。它的作用是让Python层可以像调用普通算子一样调用自定义SwiGLU。


注册侧负责检查输入是否为NPU上的float32连续张量，并生成tiling数据。启动算子内核时使用当前NPU流，测试脚本通过torch.ops.swiglu_custom.swiglu(gate, up)调用该接口。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">SwiGluTiling BuildTiling(uint32_t totalSize) { const uint32_t coreNum = std::max(1U, std::min(kDefaultBlockDim, totalSize)); SwiGluTiling tiling {}; tiling.totalSize = totalSize; tiling.coreNum = coreNum; tiling.elementsPerCore = (totalSize + coreNum - 1U) / coreNum; return tiling; } torch::Tensor swiglu_baseline_npu(const torch::Tensor &amp;gate, const torch::Tensor &amp;up) { CheckInput(gate, up); auto output = torch::empty_like(gate); const SwiGluTiling tiling = BuildTiling(static_cast<uint32_t>(gate.numel())); aclrtStream stream = c10_npu::getCurrentNPUStream().stream(); const uint32_t launchRet = ACLRT_LAUNCH_KERNEL(swiglu_baseline_kernel)( tiling.coreNum, stream, static_cast&lt;uint8_t *&gt;(gate.data_ptr()), static_cast&lt;uint8_t *&gt;(up.data_ptr()), static_cast&lt;uint8_t *&gt;(output.data_ptr()), static_cast&lt;uint8_t *&gt;(workspaceD), static_cast&lt;uint8_t *&gt;(tilingD)); TORCH_CHECK(launchRet == 0, &quot;swiglu_baseline_kernel launch failed&quot;); CHECK_ACL_THROW(aclrtSynchronizeStream(stream)); return output; } TORCH_LIBRARY(swiglu_custom, m) { m.def(&quot;swiglu(Tensor gate, Tensor up) -&gt; Tensor&quot;); } TORCH_LIBRARY_IMPL(swiglu_custom, PrivateUse1, m) { m.impl(&quot;swiglu&quot;, swiglu_baseline_npu); }</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_torch_op.py --rows 128 --hidden 1024 --atol 1e-3 --rtol 1e-3</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">./out/bin/swiglu_baseline_standalone \ --rows 128 \ --hidden 1024 \ --block-dim 8 \ --warmup 10 \ --repeat 50 \ --rounds 5</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_qwen_forward.py --batch 1 --seq 128 --hidden 896 --intermediate 4864 python3 tests/compare_qwen_native.py \ --model /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Models/Qwen2.5-0.5B \ --repeat 3 \ --attn-implementation eager</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/SwiGluBaselineExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/SwiGluBaselineExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/swiglu_baseline_standalone --rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5


## 任务拓展


完成基础实验后，可以继续围绕以下方向拓展：


• 在相同输入规模下对比基础版、优化版和原生算子的耗时。


• 调整任务划分和分块参数，观察正确性、吞吐和尾块处理是否变化。


• 将单算子测试、独立直调测试和模型替换测试的结果放在一起分析，区分算子内核内部耗时与端到端调度开销。


• 进一步尝试双缓冲、异步搬运、多级流水、FP16/BF16支持或更贴近硬件矩阵单元的实现。


## 实验总结


通过本实验，可以完整理解SwiGLU基础版算子的开发过程：从公式拆解、tiling设计、算子内核实现，到PyTorch注册和Qwen2.5替换验证。基础版代码路径直接，适合用来理解公式、索引和动态库加载之间的关系。


基础版SwiGLU把一个模型中的融合激活拆成可独立实现的算子，再完整走通从公式到PyTorch接口的开发链路。它给后续优化提供了统一的输入、统一的接口和统一的模型替换方式，因此性能对比才有清晰的参照系。
